In [ ]:
!pip install pyinaturalist -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
from pyinaturalist import get_observations

# Test query - moose in BC, research grade only
results = get_observations(
    taxon_name="Alces alces",
    place_id=7085,
    quality_grade="research",
    photos=True,
    per_page=10,
    page=1
)

print(f"Total results available: {results['total_results']}")
for obs in results['results']:
    print(obs['id'], obs['photos'][0]['url'])

Total results available: 2313
361605604 https://inaturalist-open-data.s3.amazonaws.com/photos/659949306/square.jpg
361596769 https://inaturalist-open-data.s3.amazonaws.com/photos/659931059/square.jpg
361518561 https://inaturalist-open-data.s3.amazonaws.com/photos/659774799/square.jpg
361494486 https://inaturalist-open-data.s3.amazonaws.com/photos/659729059/square.jpg
361493153 https://inaturalist-open-data.s3.amazonaws.com/photos/659725780/square.jpg
361343035 https://inaturalist-open-data.s3.amazonaws.com/photos/659446217/square.jpg
361330259 https://inaturalist-open-data.s3.amazonaws.com/photos/659420421/square.jpg
361178816 https://inaturalist-open-data.s3.amazonaws.com/photos/659130764/square.jpg
361048447 https://static.inaturalist.org/photos/658882668/square.jpg
360769864 https://inaturalist-open-data.s3.amazonaws.com/photos/658340358/square.jpg


In [ ]:
import requests
import os
from pyinaturalist import get_observations
import time

def download_species(taxon_name, place_id, species_label, max_images=500):
    os.makedirs(f"data/{species_label}", exist_ok=True)

    downloaded = 0
    page = 1
    per_page = 100
    metadata = []

    while downloaded < max_images:
        results = get_observations(
            taxon_name=taxon_name,
            place_id=place_id,
            quality_grade="research",
            photos=True,
            per_page=per_page,
            page=page
        )

        if not results['results']:
            break

        for obs in results['results']:
            if downloaded >= max_images:
                break
            try:
                photo_url = obs['photos'][0]['url'].replace("square", "large")
                obs_id = obs['id']
                filename = f"data/{species_label}/{obs_id}.jpg"

                response = requests.get(photo_url, timeout=10)
                if response.status_code == 200:
                    with open(filename, 'wb') as f:
                        f.write(response.content)
                    metadata.append({
                        'obs_id': obs_id,
                        'filename': filename,
                        'taxon': taxon_name,
                        'label': species_label,
                        'url': photo_url
                    })
                    downloaded += 1
                    if downloaded % 50 == 0:
                        print(f"{species_label}: {downloaded} images downloaded")
                time.sleep(0.5)
            except Exception as e:
                print(f"Failed on {obs_id}: {e}")
                continue

        page += 1
        time.sleep(1)

    print(f"Finished {species_label}: {downloaded} images total")
    return met

In [ ]:
import os

# Check if folder was created
if os.path.exists("data/moose"):
    files = os.listdir("data/moose")
    print(f"Files in data/moose: {len(files)}")
    print(files[:5])
else:
    print("data/moose folder does not exist")

# Check metadata
print(f"Metadata records: {len(metadata)}")

data/moose folder does not exist


NameError: name 'metadata' is not defined

In [ ]:
import requests
import os

# Test single image download
os.makedirs("data/moose", exist_ok=True)

test_url = "https://inaturalist-open-data.s3.amazonaws.com/photos/659949306/large.jpg"
response = requests.get(test_url, timeout=10)

print(f"Status code: {response.status_code}")
print(f"Content length: {len(response.content)} bytes")

if response.status_code == 200:
    with open("data/moose/test.jpg", "wb") as f:
        f.write(response.content)
    print("Image saved successfully")
else:
    print("Download failed")

Status code: 200
Content length: 770867 bytes
Image saved successfully


In [ ]:
import requests
import os
from pyinaturalist import get_observations
import time
import json

def download_species(taxon_name, place_id, species_label, max_images=50):
    os.makedirs(f"data/{species_label}", exist_ok=True)

    downloaded = 0
    page = 1
    metadata = []

    while downloaded < max_images:
        print(f"Fetching page {page}...")

        try:
            results = get_observations(
                taxon_name=taxon_name,
                place_id=place_id,
                quality_grade="research",
                photos=True,
                per_page=50,
                page=page
            )
        except Exception as e:
            print(f"API call failed: {e}")
            break

        if not results['results']:
            print("No more results.")
            break

        for obs in results['results']:
            if downloaded >= max_images:
                break
            try:
                photo_url = obs['photos'][0]['url'].replace("square", "large")
                obs_id = obs['id']
                filename = f"data/{species_label}/{obs_id}.jpg"

                response = requests.get(photo_url, timeout=10)
                if response.status_code == 200:
                    with open(filename, 'wb') as f:
                        f.write(response.content)
                    metadata.append({
                        'obs_id': obs_id,
                        'filename': filename,
                        'taxon': taxon_name,
                        'label': species_label,
                        'url': photo_url
                    })
                    downloaded += 1
                    print(f"Downloaded {downloaded}: {obs_id}.jpg")
                else:
                    print(f"Failed {obs_id}: status {response.status_code}")
                time.sleep(0.5)
            except Exception as e:
                print(f"Error on obs {obs['id']}: {e}")
                continue

        page += 1
        time.sleep(1)

    # Save metadata to JSON
    with open(f"data/{species_label}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nDone. {downloaded} images downloaded for {species_label}.")
    return metadata

# Run for moose, 50 images
metadata = download_species(
    taxon_name="Alces alces",
    place_id=7085,
    species_label="moose",
    max_images=50
)

Fetching page 1...
Downloaded 1: 361605604.jpg
Downloaded 2: 361596769.jpg
Downloaded 3: 361518561.jpg
Downloaded 4: 361494486.jpg
Downloaded 5: 361493153.jpg
Downloaded 6: 361343035.jpg
Downloaded 7: 361330259.jpg
Downloaded 8: 361178816.jpg
Downloaded 9: 361048447.jpg
Downloaded 10: 360769864.jpg
Downloaded 11: 360694827.jpg
Downloaded 12: 360378958.jpg
Downloaded 13: 360332271.jpg
Downloaded 14: 360069553.jpg
Downloaded 15: 359382079.jpg
Downloaded 16: 359242094.jpg
Downloaded 17: 359051918.jpg
Downloaded 18: 358904861.jpg
Downloaded 19: 358817113.jpg
Downloaded 20: 358728899.jpg
Downloaded 21: 358464546.jpg
Downloaded 22: 358340881.jpg
Downloaded 23: 358174634.jpg
Downloaded 24: 358167778.jpg
Downloaded 25: 358148890.jpg
Downloaded 26: 357711435.jpg
Downloaded 27: 357017442.jpg
Downloaded 28: 356943531.jpg
Downloaded 29: 356645930.jpg
Downloaded 30: 356645910.jpg
Downloaded 31: 356645899.jpg
Downloaded 32: 356645883.jpg
Downloaded 33: 356645866.jpg
Downloaded 34: 356645715.jpg
Down

In [ ]:
!pip install humanfriendly -q
!git clone https://github.com/agentmorris/MegaDetector.git
!pip install -e MegaDetector/ -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.8 MB/s eta 0:00:00
Cloning into 'MegaDetector'...
remote: Enumerating objects: 26836, done.
remote: Counting objects: 100% (1035/1035), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 26836 (delta 831), reused 776 (delta 774), pack-reused 25801 (from 4)
Receiving objects: 100% (26836/26836), 203.87 MiB | 31.81 MiB/s, done.
Resolving deltas: 100% (18119/18119), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.5/820.5 kB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.7/200.7 MB 6.3 MB/s eta 0:00:00
   ━

In [ ]:
!wget -q https://github.com/agentmorris/MegaDetector/releases/download/v5.0/md_v5a.0.0.pt -O MegaDetector/md_v5a.0.0.pt
print("Model downloaded.")

Model downloaded.


In [ ]:
!python MegaDetector/megadetector/detection/run_detector_batch.py \
    MegaDetector/md_v5a.0.0.pt \
    data/moose \
    data/moose_detections.json \
    --threshold 0.1 \
    --recursive


51 image files found in the input directory
PyTorch reports 0 available CUDA devices
GPU available: False
PyTorch reports 0 available CUDA devices
GPU available: False
2026-05-16 05:21:45.731277: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-16 05:21:46.655456: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading PT detector with compatibility mode classic
No image size available in model metadata, defaulting to 1280
Using model stride: 64
PTDetector using device cpu
Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Loaded model in 18.48

In [ ]:
import json

with open("data/moose_detections.json", "r") as f:
    detections = json.load(f)

# Summary stats
total_images = len(detections['images'])
images_with_animals = 0
images_empty = 0
low_confidence = 0

for img in detections['images']:
    animal_found = False
    for det in img.get('detections', []):
        if det['category'] == '1' and det['conf'] >= 0.5:
            animal_found = True
    if animal_found:
        images_with_animals += 1
    elif not img.get('detections'):
        images_empty += 1
    else:
        low_confidence += 1

print(f"Total images processed: {total_images}")
print(f"Images with animal detected (conf >= 0.5): {images_with_animals}")
print(f"Images with no detections: {images_empty}")
print(f"Images with only low confidence detections: {low_confidence}")

# Show first 3 detections in detail
print("\nSample detections:")
for img in detections['images'][:3]:
    print(f"\n  File: {img['file']}")
    for det in img.get('detections', []):
        print(f"    Category: {det['category']} | Confidence: {det['conf']:.2f} | BBox: {det['bbox']}")


Total images processed: 51
Images with animal detected (conf >= 0.5): 29
Images with no detections: 17
Images with only low confidence detections: 5

Sample detections:

  File: data/moose/355653629.jpg

  File: data/moose/355654615.jpg

  File: data/moose/355814813.jpg
    Category: 1 | Confidence: 0.67 | BBox: [0.4179, 0.5859, 0.1158, 0.0976]


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


CUDA available: True
Device: Tesla T4


In [3]:
import os

base = "/content/drive/MyDrive/bc-dashcam"
dirs = [
    f"{base}/data/moose",
    f"{base}/data/elk",
    f"{base}/data/bear",
    f"{base}/data/cougar",
    f"{base}/data/deer",
    f"{base}/data/coyote",
    f"{base}/data/raccoon",
    f"{base}/detections",
    f"{base}/metadata",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Directory structure created:")
for d in dirs:
    print(f"  {d}")

Directory structure created:
  /content/drive/MyDrive/bc-dashcam/data/moose
  /content/drive/MyDrive/bc-dashcam/data/elk
  /content/drive/MyDrive/bc-dashcam/data/bear
  /content/drive/MyDrive/bc-dashcam/data/cougar
  /content/drive/MyDrive/bc-dashcam/data/deer
  /content/drive/MyDrive/bc-dashcam/data/coyote
  /content/drive/MyDrive/bc-dashcam/data/raccoon
  /content/drive/MyDrive/bc-dashcam/detections
  /content/drive/MyDrive/bc-dashcam/metadata


In [4]:
!pip install pyinaturalist humanfriendly -q
!git clone https://github.com/agentmorris/MegaDetector.git
!pip install -e MegaDetector/ -q
!wget -q https://github.com/agentmorris/MegaDetector/releases/download/v5.0/md_v5a.0.0.pt -O MegaDetector/md_v5a.0.0.pt
print("All dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
Cloning into 'MegaDetector'...
remote: Enumerating objects: 26836, done.
remote: Counting objects: 100% (1037/1037), done.
remote: Compressing objects: 100% (262/262), done.
remote: Total 26836 (delta 833), reused 777 (delta 775), pack-reused 25799 (from 4)
Receiving objects: 100% (26836/26836), 203.87 MiB | 38.09 MiB/s, done.
Resolving deltas: 100% (18119/18119), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.5/820.5 kB 21.4 